# Problem Set 6: Work Training: Run an RCT, then OLS

**Economics of Health, Human Capital, and Happiness (SOC POL 333)**
Northwestern University, Fall 2026 — due 11/4 (beginning of Wednesday lecture)

---

**Name:** *[REPLACE WITH YOUR NAME — double-click this cell, edit, then press Shift+Enter]*

---

Design and analyze an RCT of a work training program, then compare with OLS.

**Before you do anything else:** go to **File → Save a copy in Drive**.

> 📝 **DRAFT for instructor review** — questions and emphasis to be finalized with Prof. Schwandt.

## Part 0: Set-up

In this problem set you will analyze the **National Supported Work (NSW) demonstration** — a real 1970s job-training experiment in which applicants were randomly assigned to receive training or not. It became one of the most famous datasets in economics when Robert LaLonde (1986) asked: *if we hadn't run the experiment, would standard regression methods have found the right answer?* You will replicate his exercise: first estimate the effect using the **RCT**, then try to recover it with **OLS** on observational data.

Variables (in both files): `treat`, `age`, `education`, `black`, `hispanic`, `married`, `nodegree` (no high-school degree), `re74`/`re75` (real earnings in 1974/75, before the program), and `re78` (real earnings in 1978, the **outcome**).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

print("Ready!")

In [ ]:
nsw = pd.read_stata("https://raw.githubusercontent.com/thetruejacob/e3h-2026-notebooks/main/data/nsw_dw.dta")          # the experiment: treated + randomized controls
cps = pd.read_stata("https://raw.githubusercontent.com/thetruejacob/e3h-2026-notebooks/main/data/cps_controls.dta")     # observational comparison group from the CPS

print("NSW experimental sample:", nsw.shape, "| treated:", int(nsw.treat.sum()))
print("CPS comparison sample  :", cps.shape)
nsw.head()

## Part 1: The experiment

### Balance

As always with an RCT, first check that randomization worked: regress each *pre-treatment* characteristic on `treat`.

In [ ]:
balance_vars = ["age", "education", "black", "hispanic", "married", "nodegree", "re74", "re75"]

balance_regs = [smf.ols(f"{v} ~ treat", nsw).fit(cov_type="HC1") for v in balance_vars]
print(summary_col(balance_regs, stars=True, model_names=balance_vars,
                  regressor_order=["treat"], drop_omitted=True,
                  info_dict={"N": lambda r: f"{int(r.nobs):,}"}))

**Question 1.** Is the sample balanced? Which differences, if any, are statistically significant, and are they large in economic terms?

**Answer:** *[your answer here]*

### The experimental estimate

Now the effect of training on 1978 earnings — first the simple comparison, then adding the pre-treatment covariates.

In [ ]:
exp1 = smf.ols("re78 ~ treat", nsw).fit(cov_type="HC1")
exp2 = smf.ols("re78 ~ treat + age + education + black + hispanic + married + nodegree + re74 + re75",
               nsw).fit(cov_type="HC1")

print(summary_col([exp1, exp2], stars=True, model_names=["(1) no controls", "(2) with controls"],
                  regressor_order=["treat"], drop_omitted=True,
                  info_dict={"N": lambda r: f"{int(r.nobs):,}"}))

**Question 2.** What is the experimental estimate of the training effect, in dollars? Why does adding covariates barely change the coefficient — and why is that exactly what randomization promises?

**Answer:** *[your answer here]*

## Part 2: Pretend there was no experiment

Now LaLonde's twist. Suppose we only had the **treated** men, and — like most researchers most of the time — no randomized control group. Instead we grab a comparison group of men from the **Current Population Survey** (CPS), a standard national survey. First compare the raw characteristics.

In [ ]:
treated = nsw[nsw.treat == 1]
obs = pd.concat([treated, cps], ignore_index=True)   # treated + CPS comparison

comparison = pd.DataFrame({
    "NSW treated": treated[balance_vars + ["re78"]].mean(),
    "CPS comparison": cps[balance_vars + ["re78"]].mean(),
}).round(2)
comparison

**Question 3.** Compare the two columns. In what ways do the CPS men differ from the NSW treated men? Are these differences things that also affect earnings?

**Answer:** *[your answer here]*

### The observational estimates

In [ ]:
obs1 = smf.ols("re78 ~ treat", obs).fit(cov_type="HC1")
obs2 = smf.ols("re78 ~ treat + age + education + black + hispanic + married + nodegree + re74 + re75",
               obs).fit(cov_type="HC1")

print(summary_col([exp1, obs1, obs2], stars=True,
                  model_names=["(1) RCT", "(2) CPS naive", "(3) CPS + controls"],
                  regressor_order=["treat"], drop_omitted=True,
                  info_dict={"N": lambda r: f"{int(r.nobs):,}"}))

**Question 4.** Column (2) suggests training massively *reduces* earnings. Explain, in terms of **selection bias**, why the naive observational comparison is so wrong, and in which direction the selection goes.

**Answer:** *[your answer here]*

**Question 5.** Column (3) controls for everything we observe — age, education, race, marital status, degree, and two years of pre-program earnings. Does it recover the experimental answer from column (1)? What does this exercise teach you about when you can (and cannot) trust OLS with controls as a substitute for an experiment?

**Answer:** *[your answer here]*

**Question 6 (optional bonus).** Restrict the CPS comparison group to men who look more like the treated group (for example: `nodegree == 1`, or `re75` below some cutoff) and re-run column (3). Does the estimate move toward the experimental one? What modern methods does this hint at?

In [ ]:
# Your bonus analysis here (optional)


## Submitting

Two steps, in this order:

1. **Submit your notebook** — type your name in the box in the cell below, then run it (▶). A Google popup will ask you to sign in: **choose your Northwestern account** (…@u.northwestern.edu). Submissions from non-Northwestern accounts are rejected. You should see "Submitted!".
2. **Submit your PDF on Canvas** — **File → Print → Save as PDF**, then upload the PDF to this week's Canvas assignment. The Canvas submission is the official one used for grading and deadlines.

(Before submitting, make sure your name is filled in at the top of the notebook and that you have run everything top to bottom via **Runtime → Run all** with no errors.)

In [ ]:
#@title ▶ Submit this notebook { display-mode: "form" }
#@markdown Type your full name, then run this cell. Sign in with your **Northwestern** Google account when prompted.
STUDENT_NAME = "" #@param {type:"string"}

SUBMIT_URL = "https://script.google.com/macros/s/AKfycbzRoLWLJHSItqAVf8GHEmKuvarpoLybC9bAsgnud5hoFDG4W-9YsrSHZl_Oo3cMnoFP/exec"
PS_ID = "PS06"

import json, subprocess, requests
from google.colab import auth

name = STUDENT_NAME.strip()
if not name or "REPLACE" in name.upper():
    raise SystemExit("⚠️ Please type your name in the box above, then run this cell again.")

# Sign in with Google (popup) and collect proof of identity for the server to verify
auth.authenticate_user()
id_token, access_token = None, None
try:
    import google.auth
    from google.auth.transport.requests import Request
    creds, _ = google.auth.default()
    creds.refresh(Request())
    id_token = getattr(creds, "id_token", None)
    access_token = getattr(creds, "token", None)
except Exception:
    pass
if not id_token:
    try:
        id_token = subprocess.run(["gcloud", "auth", "print-identity-token"],
                                  capture_output=True, text=True, timeout=30).stdout.strip() or None
    except Exception:
        pass
if not (id_token or access_token):
    raise SystemExit("⚠️ Could not verify your Google sign-in. Run the cell again; if it persists, email the TA.")

from google.colab import _message
notebook = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]

resp = requests.post(
    SUBMIT_URL,
    json={"token": "e3h-2026-nu", "name": name, "ps": PS_ID,
          "id_token": id_token or "", "access_token": access_token or "",
          "notebook": notebook},
    timeout=120,
)
result = resp.json()
if result.get("ok"):
    print(f"✅ Submitted as {result['email']} — received for {result['ps']}.")
    print("Now also upload your PDF to Canvas (File → Print → Save as PDF).")
else:
    print("❌ Submission failed:", result.get("error"))
    print("Try again in a minute; if it keeps failing, email the TA and submit on Canvas only.")